# 사람들은 왜 '인외'에 빠졌을까? 데이터로 뜯어보기

DataPopcorn 테크블로그 글의 재현 노트북입니다. `런타임 → 모두 실행`을 누르면 글에 나온 숫자와 차트 6개가 그대로 다시 만들어집니다.

- 차트 6개: Step 3(검색 추이·검색수), Step 4(급변점), Step 5(누가), Step 6(공급), Step 7(태그 DNA), Step 8(블로그 이유)
- 데이터: 2026-10-20~21에 수집한 스냅샷(네이버 데이터랩, 네이버 검색광고 키워드도구, 구글 트렌드, 리디 키워드 검색, 네이버 블로그 코딩 결과)
- API 키가 없어도 실행됩니다. 직접 다시 수집하려면 맨 아래 **부록: 수집 코드**를 보세요.

In [ ]:
# 0. 한글 폰트 설정 (Colab이면 나눔고딕 설치)
import os, sys, json, io, urllib.request
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.system('apt-get -qq install -y fonts-nanum > /dev/null')
FONT_CANDIDATES = ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf', '/System/Library/Fonts/AppleSDGothicNeo.ttc', 'C:/Windows/Fonts/malgun.ttf']
for f in FONT_CANDIDATES:
    if os.path.exists(f):
        fm.fontManager.addfont(f)
        plt.rcParams['font.family'] = fm.FontProperties(fname=f).get_name()
        break
plt.rcParams.update({'axes.unicode_minus': False, 'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
C = {'main': '#E8590C', 'sub': '#1C7ED6', 'gray': '#868E96', 'purple': '#7048E8'}

# 데이터 위치: 공개 GitHub 저장소 (로컬에서 돌릴 때는 DATA_DIR 환경변수로 폴더 지정)
BASE = os.environ.get('DATA_DIR', 'https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/inoe-popularity/data')
def read_bytes(rel):
    p = f'{BASE}/{rel}'
    if p.startswith('http'):
        return urllib.request.urlopen(p).read()
    return open(p, 'rb').read()
def load_json(rel): return json.loads(read_bytes(rel).decode('utf-8'))
def load_csv(rel, **kw): return pd.read_csv(io.BytesIO(read_bytes(rel)), **kw)
def datalab(rel):
    """데이터랩 응답 JSON → {키워드그룹: 날짜 인덱스 Series}"""
    j = load_json(rel); out = {}
    for r in j['results']:
        s = pd.Series({x['period']: x['ratio'] for x in r['data']}, dtype=float)
        s.index = pd.to_datetime(s.index); out[r['title']] = s
    return out
print('데이터 경로:', BASE)

## Step 2. 10년 검색 추이: 붐 이전에도 오르고 있었나

데이터랩은 기간 안에서 가장 많이 검색된 시점을 100으로 놓은 **상대지수**만 줍니다. 값이 없는 주(검색이 너무 적은 주)는 응답에서 빠지므로 0으로 채웁니다.

In [ ]:
w = datalab('naver_datalab/week_inoe_2016_2026.json')['인외']
w = w.reindex(pd.date_range(w.index.min(), w.index.max(), freq='W-MON'), fill_value=0)
yearly = w.groupby(w.index.year).mean().round(2)
print(yearly.loc[2016:2026])
print('2019 → 2025 배수:', round(yearly[2025] / yearly[2019], 1))

# 교차검증 1: 구글 트렌드(대한민국) 월별
g = load_csv('google_trends/gtrends_kr_inoe_month.csv', skiprows=2)
g.columns = ['month', 'google']; print(g.tail(4).to_string(index=False))
# 교차검증 2: 구글의 '인외'에는 '인외마경'(주술회전·페이트 용어)이 섞인다 → 네이버 정확일치 키워드로 분리 확인
print(*read_bytes('google_trends/gtrends_kr_inoe_related.csv').decode('utf-8-sig').splitlines()[3:10], sep=chr(10))
mg = datalab('naver_datalab/month_compare_inoe_vs_inoemagyeong.json')
print('네이버 인외마경 최고 월:', mg['인외마경'].idxmax().date(), '/ 인외 최고 월:', mg['인외'].idxmax().date())

## Step 3. 상대지수를 실제 검색 횟수로 바꿉니다

네이버 검색광고 키워드도구의 키워드 상세 화면은 최근 12개월 **월별 검색수**(PC·모바일)와 성별·연령 비중을 보여 줍니다. 화면의 차트 좌표를 숫자로 옮긴 값이라 수십 회 단위의 반올림 오차가 있습니다.

In [ ]:
ui = load_json('naver_searchad/searchad_ui_detail.json')['details']
months = pd.period_range('2025-10', '2026-09', freq='M')
abs_m = pd.Series(np.array(ui['인외']['monthly'][0]) + np.array(ui['인외']['monthly'][1]), index=months)
print(abs_m.to_string())
base_abs = abs_m[:'2026-08'].mean()
print(f'붐 직전 11개월 평균: {base_abs:,.0f}회 → 2026년 9월: {abs_m["2026-09"]:,}회 ({abs_m["2026-09"]/base_abs:.1f}배)')

d = datalab('naver_datalab/daily_inoe_2025_2026.json')['인외'].reindex(pd.date_range('2025-01-01', '2026-10-20'), fill_value=0)
k = abs_m['2026-09'] / d['2026-09'].sum()          # 지수 1 = 검색 몇 회인지 (9월 실측으로 환산)
print(f'10월 1~20일 추정 검색수: 약 {d["2026-10-01":"2026-10-20"].sum()*k:,.0f}회')

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={'width_ratios': [1.5, 1]})
yy = yearly.loc[2016:2025]
ax[0].bar(yy.index.astype(str), yy.values, color=[C['gray']]*9 + [C['main']])
ax[0].set_title('① 붐 이전에도 해마다 조금씩 올랐다 (주간 지수 연평균)', loc='left')
ax[1].bar([str(p)[2:] for p in abs_m.index], abs_m.values, color=[C['gray']]*11 + [C['main']])
ax[1].set_title('② 2026년 9월 월간 검색수', loc='left'); ax[1].tick_params(axis='x', labelsize=7)
plt.tight_layout(); plt.show()

## Step 4. 급변점: 무엇이 먼저 움직였나

같은 요청에 넣은 키워드끼리는 같은 척도(5개 중 최고일=100)라 서로 비교할 수 있습니다.

In [ ]:
B = pd.DataFrame(datalab('naver_datalab/daily_compare_lotteworld_2026.json')).reindex(pd.date_range('2026-06-01', '2026-10-20')).fillna(0)
print(B['2026-09-10':'2026-09-19'].round(1).to_string())
base = d['2025-09-01':'2026-08-31'].mean()
print(f'평소(2025.9~2026.8) 일평균 지수 {base:.1f} → 9/13~10/20 평균 {d["2026-09-13":"2026-10-20"].mean():.1f} ({d["2026-09-13":"2026-10-20"].mean()/base:.1f}배)')
print(f'최근 7일(10/14~20)은 평소의 {d["2026-10-14":"2026-10-20"].mean()/base:.1f}배, 최고일 {d.idxmax().date()}')

fig, ax = plt.subplots(figsize=(12, 4.3)); p = B['2026-08-20':]
ax.plot(p.index, p['롯데월드 다크판타지'], color=C['sub'], label='롯데월드 다크판타지·퍼레이드')
ax.plot(p.index, p['인외'], color=C['main'], lw=2.6, label='인외')
ax.plot(p.index, p['베스퍼'], color=C['purple'], ls='--', label='베스퍼·가로등 신사')
for dt in ['2026-09-12', '2026-09-18']: ax.axvline(pd.Timestamp(dt), color=C['gray'], ls=':', lw=0.8)
ax.legend(frameon=False); ax.set_title('퍼레이드(9/12) → 인외(9/13) → 캐릭터 이름(9/16)', loc='left'); plt.show()

## Step 5. 누가 검색하나, 누가 새로 들어왔나

① 현재 비중은 키워드도구의 성별·연령 막대(PC·모바일)를 최근 30일 검색량으로 가중평균합니다.
② 증가 배수는 데이터랩 연령 필터 결과를, 같은 요청에 넣은 **기준어**(유튜브, 날씨) 대비 비율로 바꾼 뒤 붐 전후를 나눈 값입니다. 기준어마다 연령 구성이 달라 절대 비중은 알 수 없고, "어느 층이 더 많이 늘었나"만 비교합니다.

In [ ]:
wpc, wmo = 2550, 15500     # 최근 30일 PC·모바일 검색수 (검색광고 API)
gb, ab = ui['인외']['bars']
fem = (gb[0][1]*wpc + gb[1][1]*wmo) / (wpc + wmo); male = (gb[0][0]*wpc + gb[1][0]*wmo) / (wpc + wmo)
print(f'여성 비중: {fem/(fem+male)*100:.1f}%')
ages = ['0-12', '13-19', '20-24', '25-29', '30-39', '40-49', '50-']
age = np.array([(ab[0][i]*wpc + ab[1][i]*wmo) / (wpc + wmo) for i in range(7)]); age = age / age.sum() * 100
print(dict(zip(ages, age.round(1))))

dd = pd.DataFrame(load_json('naver_datalab/demo_anchor.json')['rows'])
pre, post = '2025-01-01~2025-08-31', '2026-09-01~2026-10-20'
res = {}
for anchor in ['youtube', 'weather']:
    t = dd.assign(r=dd.inoe / dd[anchor]).pivot(index='filter', columns='period', values='r')
    res[anchor] = t[post] / t[pre]
mult = pd.DataFrame(res); mult['기하평균'] = np.sqrt(mult.youtube * mult.weather)
order = ['13-18', '19-24', '25-29', '30-34', '35-39', '40-44', '45-49', '여성', '남성']
print(mult.loc[order].round(1))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].bar(ages, age, color=[C['gray'], C['gray'], C['main'], C['main'], C['main'], C['gray'], C['gray']])
for i, v in enumerate(age): ax[0].text(i, v + 0.6, f'{v:.0f}%', ha='center', fontsize=9)
ax[0].set_title(f'① 지금 검색자: 여성 {fem/(fem+male)*100:.0f}% · 연령대 비중', loc='left')
ages2 = order[:7]; m2 = mult.loc[ages2, '기하평균']
ax[1].bar(ages2, m2.values, color=[C['gray'], C['gray'], C['main'], C['main'], C['main'], C['gray'], C['gray']])
for i, v in enumerate(m2.values): ax[1].text(i, v + 0.4, f'{v:.0f}배', ha='center', fontsize=9)
ax[1].set_title('② 붐 이후 증가 배수 (기준어 대비, 기하평균)', loc='left')
plt.tight_layout(); plt.show()

## Step 6. 공급: 웹소설 시장에는 인외가 얼마나 쌓여 있었나

리디 로맨스·로판 키워드 검색에서 `인외존재` 태그 작품 전체와 비교군 `회귀/타임슬립` 태그 작품 전체를 모으고, 작품마다 전자책 출간일을 붙였습니다. 태그가 생기기 전 작품은 덜 잡히므로 두 태그의 **비율**을 2023년부터만 해석합니다.

In [ ]:
rb = load_csv('ridi/ridi_books.csv', parse_dates=['date'])
rb['year'] = rb.date.dt.year
t = rb.pivot_table(index='year', columns='tag', values='id', aggfunc='count').fillna(0).astype(int).loc[2019:2026]
t['인외/회귀'] = (t['인외존재'] / t['회귀/타임슬립']).round(2)
print(t)
for y in [2025, 2026]:
    print(y, '1~9월 출간:', rb[(rb.year == y) & (rb.date.dt.month <= 9)].groupby('tag').size().to_dict())
print('성인 전용 비율:', rb.groupby('tag').adult.mean().round(3).to_dict())

fig, ax = plt.subplots(figsize=(11, 4.3)); x = np.arange(len(t)); wd = 0.38
ax.bar(x - wd/2, t['인외존재'], wd, color=C['main'], label='인외존재')
ax.bar(x + wd/2, t['회귀/타임슬립'], wd, color=C['gray'], label='회귀/타임슬립(비교군)')
ax.set_xticks(x, [str(i) for i in t.index]); ax.axvspan(-0.5, 3.5, color='#f1f3f5', zorder=0)
ax2 = ax.twinx(); r = t['인외/회귀']; sel = r.index >= 2023
ax2.plot(x[sel], r[sel], color=C['purple'], marker='o', lw=2, label='인외/회귀 비율'); ax2.set_ylim(0, 2.4)
ax.legend(loc='upper left', frameon=False); ax.set_title('리디 로맨스·로판 출간 연도별 작품 수 (회색 구간은 태그 도입 전후라 해석 제외)', loc='left')
plt.tight_layout(); plt.show()

## Step 7. 인외 로맨스의 '태그 DNA'

`배수 = (해당 태그 작품 중 인외존재 비율) ÷ (그 태그 그룹 전체 평균 인외존재 비율)`. 1보다 크면 인외 작품에 유난히 많이 붙는 태그입니다.

In [ ]:
co = pd.DataFrame(load_json('ridi/ridi_tag_cooccur.json')['rows'])
for gname in ['남자 주인공', '관계']:
    s = co[co.group == gname].copy()
    s['배수'] = (s['with인외'] / s['all']) / (s['with인외'].sum() / s['all'].sum())
    s = s[s['all'] >= 500].sort_values('배수', ascending=False)
    print(f'--- {gname} (상위 7 / 하위 5)'); print(pd.concat([s.head(7), s.tail(5)])[['tag', 'with인외', 'all', '배수']].round(2).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for a_, gname in zip(axes, ['남자 주인공', '관계']):
    s = co[co.group == gname].copy(); s['배수'] = (s['with인외'] / s['all']) / (s['with인외'].sum() / s['all'].sum())
    s = s[s['all'] >= 500].sort_values('배수'); sel = pd.concat([s.head(6), s.tail(8)])
    a_.barh(sel.tag, sel['배수'], color=[C['main'] if v > 1 else C['gray'] for v in sel['배수']]); a_.axvline(1, color='#333', lw=0.8)
    for y, v in enumerate(sel['배수']): a_.text(v + 0.03, y, f'{v:.2f}배', va='center', fontsize=9)
    a_.set_title(f'{gname} 태그: 인외 작품에 붙는 배수 (1 = 평균)', loc='left')
plt.tight_layout(); plt.show()

## Step 8. 블로그 1,047건에서 사람들이 말하는 이유

원문 저작권 때문에 본문은 싣지 않고, 글마다 단어 사전으로 분류한 결과(0/1)와 주소만 공개합니다. 분류 방법은 블로그 글 Step 8과 부록을 보세요.

In [ ]:
bc = load_csv('blog/blog_coded.csv')
print(bc.groupby('period').size())
ctx = [c for c in bc if c.startswith('ctx_')]; rs = [c for c in bc if c.startswith('r_')]
print((bc.groupby('period')[ctx].mean() * 100).round(1).T)
share = (bc.groupby('period')[rs].mean() * 100).round(1).T
share['변화(%p)'] = share.iloc[:, 1] - share.iloc[:, 0]
print(share.sort_values('변화(%p)', ascending=False))

sh = share.sort_values('변화(%p)'); pre, post = sh.iloc[:, 0].values, sh.iloc[:, 1].values; y = np.arange(len(sh))
fig, ax = plt.subplots(figsize=(11, 5))
for i in range(len(sh)): ax.plot([pre[i], post[i]], [i, i], color='#ced4da', lw=3, zorder=1)
ax.scatter(pre, y, color=C['gray'], s=60, zorder=2, label='붐 이전 (~9/11)'); ax.scatter(post, y, color=C['main'], s=60, zorder=2, label='붐 이후 (9/12~)')
for i in range(len(sh)): ax.text(max(pre[i], post[i]) + 0.8, i, f'{pre[i]:.0f}% → {post[i]:.0f}%', va='center', fontsize=9)
ax.set_yticks(y, [c[2:] for c in sh.index]); ax.legend(frameon=False, loc='lower right')
ax.set_title('"인외" 언급 주변에 해당 표현이 나온 글의 비율', loc='left'); plt.tight_layout(); plt.show()

# 사전 분류 정확도 점검 표본 (직접 읽고 판정, 1=의도한 의미)
v = load_csv('blog/validation_sample.csv'); print(v.groupby('category').correct.agg(['sum', 'count']), '전체', v.correct.sum(), '/', len(v))

## 부록: 직접 다시 수집하기

아래 셀은 기본으로 꺼져 있습니다(`RUN_COLLECT = False`). 네이버 개발자센터에서 **데이터랩(검색어트렌드)**과 **검색** API를 추가한 애플리케이션을 만들고, Colab 왼쪽 열쇠 아이콘(보안 비밀)에 `NAVER_CLIENT_ID`, `NAVER_CLIENT_SECRET`을 넣은 뒤 `True`로 바꾸세요. 키를 코드에 직접 쓰지 마세요.

In [ ]:
RUN_COLLECT = False
import requests, re, time

def naver_keys():
    try:
        from google.colab import userdata
        return userdata.get('NAVER_CLIENT_ID'), userdata.get('NAVER_CLIENT_SECRET')
    except Exception:
        return os.environ.get('NAVER_CLIENT_ID'), os.environ.get('NAVER_CLIENT_SECRET')

def datalab_search(groups, start, end, unit='month', **filters):
    """groups 예: {'인외': ['인외'], '베스퍼': ['베스퍼', '가로등 신사']}  filters 예: gender='f', ages=['3']"""
    cid, sec = naver_keys()
    body = {'startDate': start, 'endDate': end, 'timeUnit': unit,
            'keywordGroups': [{'groupName': k, 'keywords': v} for k, v in groups.items()], **filters}
    r = requests.post('https://openapi.naver.com/v1/datalab/search', json=body,
                      headers={'X-Naver-Client-Id': cid, 'X-Naver-Client-Secret': sec}, timeout=20)
    r.raise_for_status(); return r.json()

UA = {'User-Agent': 'Mozilla/5.0'}
def ridi_page(tags, page=1):
    """tags: [(3702, '인외존재'), (2648, '집착남')] → (전체 작품 수, 이 페이지 작품 ID 20개)"""
    q = '&'.join(f'tag_ids={i}-{requests.utils.quote(n)}' for i, n in tags)
    h = requests.get(f'https://ridibooks.com/keyword-finder/romance?set_id=1&{q}&page={page}', headers=UA, timeout=20).text
    nd = json.loads(re.search(r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>', h, re.S).group(1))
    qq = next(x for x in nd['props']['pageProps']['dehydratedState']['queries'] if x['queryKey'][0] == 'KeywordFinderBooks')
    kb = qq['state']['data']['keywordFinderBooks']
    return kb['totalItemCount'], [it['bookShell']['book']['id'] for it in kb['items']]

def ridi_tag_count(tags):
    return ridi_page(tags)[0]

def ridi_tag_books(tag_id, name, sleep=0.3):
    """태그 작품 ID 전체 → 작품별 출간일·성인 여부"""
    total, ids, page = None, [], 1
    while True:
        total, got = ridi_page([(tag_id, name)], page)
        if not got: break
        ids += got; page += 1; time.sleep(sleep)
    rows = []
    for b in dict.fromkeys(ids):
        j = requests.get(f'https://book-api.ridibooks.com/books/{b}', headers=UA, timeout=20).json()
        rows.append({'id': b, 'ebook_publish': j['publish'].get('ebook_publish'), 'adult': j['property']['is_adult_only']})
        time.sleep(sleep / 3)
    print(f'{name}: 사이트 표시 {total}종, 수집 {len(rows)}종')
    return pd.DataFrame(rows)

def blog_search(q, sort='date'):
    """네이버 검색 API(블로그): 검색어당 최대 1,000건"""
    cid, sec = naver_keys(); out = []
    for start in range(1, 1001, 100):
        r = requests.get('https://openapi.naver.com/v1/search/blog.json', params={'query': q, 'display': 100, 'start': start, 'sort': sort},
                         headers={'X-Naver-Client-Id': cid, 'X-Naver-Client-Secret': sec}, timeout=20)
        r.raise_for_status(); items = r.json()['items']; out += items
        if len(items) < 100: break
        time.sleep(0.15)
    return out

def blog_body(link):
    """네이버 블로그 본문(모바일 페이지의 se-text-paragraph 단락)"""
    m = re.search(r'blog\.naver\.com/([^/?]+)/(\d+)', link)
    if not m: return None
    h = requests.get(f'https://m.blog.naver.com/PostView.naver?blogId={m.group(1)}&logNo={m.group(2)}', headers={'User-Agent': 'Mozilla/5.0 (iPhone)'}, timeout=15).text
    ps = [re.sub(r'<[^>]+>', ' ', p) for p in re.findall(r'<p class="se-text-paragraph[^"]*"[^>]*>([\s\S]*?)</p>', h)]
    import html as _h; return re.sub(r'\s+', ' ', _h.unescape(' '.join(ps))).strip()

MENT = r'인외|오브젝트\s?헤드|가로등\s?신사'
TAGTOK = r'#\S+|(?:다정|헌신|집착|순정|짝사랑|사랑꾼|절륜|직진|능글|츤데레|대형견|존댓말|미남|미인|연하|연상|순진|평범|상처|능력|애교|조신|유혹|동정|오만|계략|무심|철벽|냉정|까칠|후회|카리스마|재벌|사이다|뇌섹|사차원|인외|강)(?:공|수|남|녀)(?![가-힣])'
REASON = {'빈칸·상상': r'상상|투영|빈칸|여백|표정이?\s?없|얼굴이?\s?없|얼굴이 안|이목구비|해석의|어떤 표정|무슨 생각',
          '갭·반전': r'반전 매력|반전|갭모에|갭|다정|젠틀|신사적|매너|스윗|섬세|조심스',
          '이질감·기묘함': r'기괴|기묘|낯설|이질|으스스|그로테스크|섬뜩',
          '압도·초월': r'압도|초월|거대|덩치|체구|피지컬|키가 2|강대|위압',
          '순애·헌신': r'순애|일편단심|헌신|평생|각인|나만|너만|오직 (?:그녀|여주|한 사람)|반려',
          '인간 피로': r'현실 남자|인간 남자|사람보다|인간보다|현실 연애|사람 말고|인간이 싫|인간혐오|인간 혐오|인간 남주',
          '2차 창작·덕질': r'팬아트|2차\s?창작|이차\s?창작|코스프레|덕질|입덕|최애|굿즈|팬덤',
          'SNS·숏폼 확산': r'릴스|쇼츠|숏폼|틱톡|알고리즘|트위터|SNS|조회수|바이럴|짤'}

def classify(text, w=150):
    """'인외' 언급 앞뒤 150자 창에서 태그 단어를 지운 뒤 이유 사전 매칭"""
    win = ' ‖ '.join(text[max(0, m.start() - w):m.end() + w] for m in re.finditer(MENT, text))
    win = re.sub(TAGTOK, ' ', win)
    return {k: bool(re.search(p, win)) for k, p in REASON.items()}

BLOG_QUERIES = ['인외콤', '인외 캐릭터', '인외 매력', '인외남주', '인외 로판', '인외 웹툰', '인외 좋아', '오브젝트헤드', '인외 남주', '인외 취향', '인외존재', '베스퍼 롯데월드', '가로등 신사']

if RUN_COLLECT:
    # 블로그 재수집 예시 (전체는 13개 검색어 × 2개 정렬, 본문 1천여 건이라 수십 분 걸립니다)
    items = {i['link']: i for q in BLOG_QUERIES[:1] for srt in ['date', 'sim'] for i in blog_search(q, srt)}
    rows = []
    for link, it in list(items.items())[:30]:
        body = blog_body(link)
        if body and len(body) >= 200 and re.search(MENT, body):
            rows.append({'date': it['postdate'], **classify(body)})
    print(pd.DataFrame(rows).mean(numeric_only=True).round(2))
    print(datalab_search({'인외': ['인외']}, '2016-01-01', '2026-10-20', 'week')['results'][0]['data'][-3:])
    print('인외존재:', ridi_tag_count([(3702, '인외존재')]))